# 14 · Regular Expressions

A **regular expression** (regex) is a mini-language for matching patterns in
text. When data isn't cleanly delimited — log lines, free-text fields, IDs
embedded in strings — regex is how you extract and validate it. Python's `re`
module is the tool.

## Always use raw strings

Write patterns as **raw strings** (`r'...'`) so backslashes mean regex
metacharacters, not Python escapes. `\d` = digit, `\w` = word char, `\s` =
whitespace; `+` = one or more, `*` = zero or more, `?` = optional.

In [ ]:
import re

text = 'Order ORD-2024-0042 shipped'
m = re.search(r'ORD-\d{4}-\d{4}', text)   # find pattern anywhere
print('found:', m.group())
print('span :', m.span())

## `match` vs `search` vs `fullmatch` vs `findall`

- `match` — anchored at the **start** of the string
- `search` — anywhere in the string (most common)
- `fullmatch` — the **whole** string must match (great for validation)
- `findall` — every non-overlapping match, as a list

In [ ]:
import re

print(bool(re.match(r'\d+', '42 apples')))       # True: starts with digits
print(bool(re.match(r'\d+', 'apples 42')))       # False: doesn't start
print(bool(re.search(r'\d+', 'apples 42')))      # True: found later
print(bool(re.fullmatch(r'\d{4}', '2024')))      # True: exactly 4 digits
print(re.findall(r'\d+', 'a1 b22 c333'))          # ['1', '22', '333']

## Capturing groups

Parentheses `(...)` **capture** parts of a match so you can pull out fields.
`group(0)` is the whole match; `group(1)`, `group(2)`, ... are the captures.

In [ ]:
import re

code = 'ORD-2024-0042'
m = re.match(r'(\w+)-(\d{4})-(\d+)', code)
print('all groups:', m.groups())
print('type :', m.group(1))
print('year :', int(m.group(2)))
print('seq  :', int(m.group(3)))

## Named groups (readable extraction)

`(?P<name>...)` names a capture so you access it by key — far clearer than
numeric positions when parsing structured text like log lines.

In [ ]:
import re

log = '2024-06-01 14:30:05 ERROR database timeout'
pat = r'(?P<date>\d{4}-\d\d-\d\d) (?P<time>\d\d:\d\d:\d\d) (?P<level>\w+) (?P<msg>.*)'
m = re.match(pat, log)
print(m.group('level'), '->', m.group('msg'))
print(m.groupdict())

## `sub` — find and replace / redact

`re.sub` replaces every match. Use it to clean, normalize, or **redact**
sensitive data (a common data-governance task).

In [ ]:
import re

messy = 'Phone:  555-123-4567 , Alt: 555.987.6543'
digits_only = re.sub(r'[^0-9]', '', 'ID: 00-1234')      # strip non-digits
print('cleaned id:', digits_only)

redacted = re.sub(r'\d{3}[-.]\d{3}[-.]\d{4}', '[REDACTED]', messy)
print(redacted)

## Compile once, reuse many times

When applying the same pattern to many rows, `re.compile` it once — clearer and
faster in a hot loop.

In [ ]:
import re

email_re = re.compile(r'^[\w.+-]+@[\w-]+\.[\w.-]+$')

for addr in ['ava.smith@example.com', 'not-an-email', 'x@y.z']:
    print(f'{addr:25s} valid={bool(email_re.fullmatch(addr))}')

> **Keep regex readable.** For anything complex, add comments with
> `re.VERBOSE`, and remember: if the data is truly tabular, the `csv` module is
> safer than a regex. Reach for regex when structure lives *inside* a text field.

### Recap

`re` matches patterns in text; use raw strings (`r'...'`); `match`/`search`/
`fullmatch`/`findall` differ by where they anchor; `(...)` captures and
`(?P<name>...)` names captures; `sub` replaces/redacts; `compile` once for reuse.
Next: JSON and serialization.